In [ ]:
!pip install -q transformers peft datasets evaluate seqeval torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!wget https://www.dropbox.com/scl/fi/ln64xjqlwd4665u9sp2bq/Full_Dataset.rar?rlkey=g5aa4zefy2xpj98n3x8ihjcp2&st=bg907w5p&dl=0
!mv Full_Dataset.rar?rlkey=g5aa4zefy2xpj98n3x8ihjcp2 Full_Dataset.rar
!unrar x /content/Full_Dataset.rar

Streaming output truncated to the last 5000 lines.
Extracting  Full_Dataset/55.txt                                           72%  OK 
Extracting  Full_Dataset/550.txt                                          72%  OK 
Extracting  Full_Dataset/5500.txt                                         72%  OK 
Extracting  Full_Dataset/5501.txt                                         72%  OK 
Extracting  Full_Dataset/5502.txt                                         72%  OK 
Extracting  Full_Dataset/5503.txt                                         72%  OK 
Extracting  Full_Dataset/5504.txt                                         72%  OK 
Extracting  Full_Dataset/5505.txt                                         72%  OK 
Extracting  Full_Dataset/5506.txt                                         72%  OK 
Extracting  Full_Dataset/5507.txt                                         72%  OK 
Extracting  Full_Dataset/5508

In [ ]:
import os
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# 1. بارگذاری توکنایزر مدل پایه
model_checkpoint = "HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. تعریف تابع برای هم‌ترازی توکن‌ها و برچسب‌ها (Label Alignment)
# وقتی BERT یک کلمه را به دو بخش تقسیم می‌کند، فقط بخش اول برچسب می‌گیرد و بقیه با 100- نادیده گرفته می‌شوند
def tokenize_and_align_labels(examples):
    # کلمات را به توکنایزر می‌دهیم
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=256
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        # پیدا کردن مپینگ بین کلمات اصلی و توکن‌های جدید
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # توکن‌های ویژه (مثل [CLS] و [SEP]) در محاسبه Loss نادیده گرفته می‌شوند (برچسب 100-)
            if word_idx is None:
                label_ids.append(-100)
            # اگر اولین توکن از یک کلمه باشد، برچسب اصلی آن را قرار می‌دهیم
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # اگر زیرکلمه‌های بعدی از همان کلمه باشند، نادیده گرفته می‌شوند
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenizer loaded and alignment function is ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded and alignment function is ready!


In [ ]:
import os
import glob
from datasets import Dataset, DatasetDict

folder_path = '/content/Full_Dataset' # مسیری که فایل‌های txt در آن اکسترکت شده‌اند

def load_clean_dataset(folder_path):
    all_tokens = []
    all_ner_tags = []

    file_list = glob.glob(os.path.join(folder_path, '*.txt'))

    for file_path in file_list:


        tokens = []
        tags = []

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip()
                if not line:
                    continue

                parts = line.split()
                if len(parts) >= 2:
                    word = parts[0]
                    tag = int(parts[-1]) # برچسب از قبل 0 یا 1 است، فقط به عدد تبدیل می‌کنیم

                    tokens.append(word)
                    tags.append(tag)

        if tokens:
            all_tokens.append(tokens)
            all_ner_tags.append(tags)

    return Dataset.from_dict({'tokens': all_tokens, 'ner_tags': all_ner_tags})

# 1. بارگذاری کل داده‌ها
full_dataset = load_clean_dataset(folder_path)
print(f"تعداد کل جملات دیتاست: {len(full_dataset)}")

# 2. تقسیم‌بندی داده‌ها (70% Train, 15% Validation, 15% Test)
# ابتدا 30 درصد جدا می‌کنیم برای تست و اعتبارسنجی
train_testvalid = full_dataset.train_test_split(test_size=0.3, seed=42)
# سپس آن 30 درصد را نصف می‌کنیم
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

raw_datasets = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

print(raw_datasets)

تعداد کل جملات دیتاست: 18269
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 12788
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2740
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2741
    })
})


In [ ]:
# اعمال توکنایزر روی تمام بخش‌های دیتاست به صورت دسته‌ای (batched)
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

# بررسی نتیجه برای اولین داده آموزشی
print("\nنمونه توکنایز شده:")
print("Tokens:", tokenized_datasets['train'][0]['tokens'])
print("Original Tags:", tokenized_datasets['train'][0]['ner_tags'])
print("Aligned Labels:", tokenized_datasets['train'][0]['labels'])

Map:   0%|          | 0/12788 [00:00<?, ? examples/s]

Map:   0%|          | 0/2740 [00:00<?, ? examples/s]

Map:   0%|          | 0/2741 [00:00<?, ? examples/s]


نمونه توکنایز شده:
Tokens: ['آقای', 'رئیس', '،', 'خانمها', 'و', 'آقایان', 'در', 'گفتگوی', 'میان', 'فرهنگها', 'و', 'تمدنها', 'بی\u200cشک', 'باید', 'هنرمندان', 'بزرگ', 'در', 'کنار', 'فیلسوفان', 'و', 'دانشمندان', 'و', 'یزدان\u200cشناسان', 'به', 'مقام', 'لایق', 'و', 'فاخر', 'خود', '،', 'دست', 'یابند', '.']
Original Tags: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Aligned Labels: [-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]


In [ ]:
from transformers import AutoModelForTokenClassification
from peft import get_peft_model, LoraConfig, TaskType

# 1. تعریف دیکشنری برچسب‌ها
id2label = {0: "NON_ENTITY", 1: "ENTITY"}
label2id = {"NON_ENTITY": 0, "ENTITY": 1}

# 2. بارگذاری مدل پایه
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# 3. پیکربندی LoRA (اصلاح کلمه TOKEN_CLS)
peft_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, # <--- این بخش اصلاح شد
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    modules_to_save=["classifier"]
)

# 4. اعمال LoRA روی مدل
peft_model = get_peft_model(model, peft_config)

# چاپ اطلاعات پارامترهای مدل
peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                   

trainable params: 591,362 || all params: 162,843,652 || trainable%: 0.3631


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

# 1. ابزاری برای هم‌اندازه کردن (Padding) طول جملات در هر بچ (Batch)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. تعریف تابع محاسبه متریک‌ها
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=2)

    # حذف توکن‌های ویژه (مثل [CLS] و زیرکلمه‌های اضافی که برچسب 100- دارند) از محاسبه
    true_predictions = [
        [p for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [l for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # یکپارچه کردن لیست‌ها برای محاسبه متریک نهایی
    flat_true_predictions = [item for sublist in true_predictions for item in sublist]
    flat_true_labels = [item for sublist in true_labels for item in sublist]

    return {
        "precision": precision_score(flat_true_labels, flat_true_predictions, zero_division=0),
        "recall": recall_score(flat_true_labels, flat_true_predictions, zero_division=0),
        "f1": f1_score(flat_true_labels, flat_true_predictions, zero_division=0),
        "accuracy": accuracy_score(flat_true_labels, flat_true_predictions),
    }

# 3. تنظیمات آموزش (Hyperparameters)
training_args = TrainingArguments(
    output_dir="./ner_binary_lora_model",
    learning_rate=2e-4,             # در روش LoRA معمولاً نرخ یادگیری بالاتری نیاز داریم
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=10,             # 3 دوره (Epoch) روی کل داده‌ها آموزش می‌بیند
    weight_decay=0.01,
    eval_strategy ="epoch",    # ارزیابی در پایان هر Epoch
    save_strategy="epoch",          # ذخیره مدل در پایان هر Epoch
    load_best_model_at_end=True,    # در نهایت بهترین مدل را بارگذاری کن
    logging_steps=50,
)

# 4. تعریف کلاس ترینر
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class =tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 🚀 5. شروع آموزش!
print("Starting the training process...")
trainer.train()

Starting the training process...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.012700,0.012275,0.914972,0.917516,0.916242,0.995836
2,0.009877,0.010360,0.945107,0.917516,0.931107,0.996630
3,0.008689,0.009710,0.932904,0.940686,0.936779,0.996848
4,0.008139,0.009278,0.944186,0.940686,0.942433,0.997147
5,0.006586,0.009362,0.943414,0.942539,0.942976,0.997170
6,0.006885,0.008991,0.937187,0.954124,0.945580,0.997274
7,0.006430,0.008734,0.939918,0.956905,0.948335,0.997412
8,0.005615,0.008876,0.941123,0.955514,0.948264,0.997412
9,0.004635,0.009065,0.940346,0.956905,0.948553,0.997424
10,0.004841,0.008973,0.942360,0.954588,0.948435,0.997424


TrainOutput(global_step=2000, training_loss=0.011723490774631501, metrics={'train_runtime': 1850.7556, 'train_samples_per_second': 69.096, 'train_steps_per_second': 1.081, 'total_flos': 7049821941016800.0, 'train_loss': 0.011723490774631501, 'epoch': 10.0})

In [ ]:
# ارزیابی مدل روی داده‌های تست (Test Dataset)
print("در حال ارزیابی مدل روی داده‌های آزمون (Test)...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n--- نتایج نهایی روی داده‌های تست ---")
print(f"Test Precision: {test_results['eval_precision']:.4f}")
print(f"Test Recall: {test_results['eval_recall']:.4f}")
print(f"Test F1-Score: {test_results['eval_f1']:.4f}")
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")

در حال ارزیابی مدل روی داده‌های آزمون (Test)...



--- نتایج نهایی روی داده‌های تست ---
Test Precision: 0.9419
Test Recall: 0.9548
Test F1-Score: 0.9483
Test Accuracy: 0.9974


In [ ]:
import torch

# جمله تستی خود را اینجا بنویسید
sample_text = "پروفسور مجید سمیعی در شهر رشت متولد شد و علی دایی نیز از دوستان او است."

# توکنایز کردن جمله و انتقال به همان پردازنده‌ای که مدل روی آن است (GPU/CPU)
inputs = tokenizer(sample_text, return_tensors="pt").to(peft_model.device)

# دریافت پیش‌بینی‌های مدل بدون محاسبه گرادیان (برای سرعت بیشتر)
with torch.no_grad():
    outputs = peft_model(**inputs)

# استخراج برچسب‌هایی که بیشترین احتمال را دارند
predictions = torch.argmax(outputs.logits, dim=2)

# تبدیل شناسه‌های توکن به کلمات واقعی
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_labels = [peft_model.config.id2label[t.item()] for t in predictions[0]]

print(f"جمله ورودی: {sample_text}\n")
print("-" * 30)
# چاپ کلمات و برچسب آن‌ها (رد کردن توکن‌های ویژه مثل [CLS] و [SEP])
for token, label in zip(tokens, predicted_labels):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        # تبدیل ENTITY به 1 و NON_ENTITY به 0 برای نمایش مطابق خواسته پروژه
        display_label = "1 (موجودیت)" if label == "ENTITY" else "0"
        print(f"{token:15} : {display_label}")

جمله ورودی: پروفسور مجید سمیعی در شهر رشت متولد شد و علی دایی نیز از دوستان او است.

------------------------------
پروفسور         : 0
مجید            : 1 (موجودیت)
سمیعی           : 1 (موجودیت)
در              : 0
شهر             : 0
رشت             : 0
متولد           : 0
شد              : 0
و               : 0
علی             : 1 (موجودیت)
دایی            : 1 (موجودیت)
نیز             : 0
از              : 0
دوستان          : 0
او              : 0
است             : 0
.               : 0


In [ ]:
# ذخیره وزن‌های LoRA در یک پوشه
save_directory = "./final_ner_lora_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"مدل و توکنایزر با موفقیت در پوشه {save_directory} ذخیره شدند.")

مدل و توکنایزر با موفقیت در پوشه ./final_ner_lora_model ذخیره شدند.


In [ ]:
import os

# Define the directory to be zipped and the desired output zip file name
save_directory = "./final_ner_lora_model"
zip_file_name = "final_ner_lora_model.zip"

# Ensure the directory exists before trying to zip it
if os.path.exists(save_directory):
  !zip -r {zip_file_name} {save_directory}
  print(f"Model zipped successfully to {zip_file_name}")
else:
  print(f"Error: Directory {save_directory} does not exist.")

  adding: final_ner_lora_model/ (stored 0%)
  adding: final_ner_lora_model/training_args.bin (deflated 53%)
  adding: final_ner_lora_model/adapter_model.safetensors (deflated 7%)
  adding: final_ner_lora_model/tokenizer_config.json (deflated 42%)
  adding: final_ner_lora_model/tokenizer.json (deflated 72%)
  adding: final_ner_lora_model/README.md (deflated 66%)
  adding: final_ner_lora_model/adapter_config.json (deflated 58%)
Model zipped successfully to final_ner_lora_model.zip


In [ ]:
# 1. نصب کتابخانه مورد نیاز (اگر نصب نیست) و لاگین
!pip install -q huggingface_hub
from huggingface_hub import notebook_login

# این دستور یک کادر لاگین در خروجی سلول باز می‌کند
notebook_login()

In [ ]:
# 2. ارسال مدل و توکنایزر به فضای ابری
# ⚠️ مهم: به جای 'your_username' نام کاربری اکانت هاگینگ‌فیس خودتان را بنویسید!
# نام مدل (repo_name) را می‌توانید به دلخواه تغییر دهید.

hf_username = "amirhosseinesbati"
repo_name = f"{hf_username}/parsbert-ner-lora-binary"

print(f"در حال ارسال مدل به مخزن: {repo_name} ...")

# ارسال وزن‌های آموزش دیده LoRA
peft_model.push_to_hub(repo_name)

# ارسال توکنایزر
tokenizer.push_to_hub(repo_name)

print("🎉 مدل با موفقیت در Hugging Face ذخیره شد!")
print(f"لینک مدل شما: https://huggingface.co/{repo_name}")

در حال ارسال مدل به مخزن: amirhosseinesbati/parsbert-ner-lora-binary ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  24%|##3       |  560kB / 2.37MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

🎉 مدل با موفقیت در Hugging Face ذخیره شد!
لینک مدل شما: https://huggingface.co/amirhosseinesbati/parsbert-ner-lora-binary
